# Remove Comments

This program uses tkinter to provide an interface for selecting and saving pdfs. The program will clear the array of annotations within the pdf. 

This is useful if a pdf with non-flattened annotations like comments needs to be cleared to obtain a clean file. 

In [3]:
import tkinter as tk
from tkinter import filedialog, messagebox
from pathlib import Path

from PyPDF2 import PdfReader, PdfWriter


In [4]:
def pick_input_pdf(title="Select input PDF"):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    path = filedialog.askopenfilename(
        title=title,
        filetypes=[("PDF files", "*.pdf")],
    )

    root.destroy()
    return path if path else None


def pick_output_pdf(default_name, title="Save output PDF as"):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    path = filedialog.asksaveasfilename(
        title=title,
        defaultextension=".pdf",
        initialfile=default_name,
        filetypes=[("PDF files", "*.pdf")],
    )

    root.destroy()
    return path if path else None



In [10]:
def remove_collapsible_comments(input_pdf: str, output_pdf: str) -> dict:
    """
    Removes collapsible PDF comment annotations:
      - /Subtype /Text   (sticky note comments)
      - /Subtype /Popup  (associated popups)

    Pure PyPDF2 implementation (Anaconda-safe).
    """
    from PyPDF2 import PdfReader, PdfWriter
    from PyPDF2.generic import NameObject, ArrayObject

    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    removed_text = 0
    removed_popup = 0

    ANNOTS = NameObject("/Annots")
    SUBTYPE = NameObject("/Subtype")
    TEXT = NameObject("/Text")
    POPUP = NameObject("/Popup")

    for page in reader.pages:
        if ANNOTS in page:
            new_annots = []

            for annot_ref in page[ANNOTS]:
                annot = annot_ref.get_object()
                subtype = annot.get(SUBTYPE)

                if subtype == TEXT:
                    removed_text += 1
                    continue

                if subtype == POPUP:
                    removed_popup += 1
                    continue

                new_annots.append(annot_ref)

            if new_annots:
                page[ANNOTS] = ArrayObject(new_annots)
            else:
                del page[ANNOTS]

        writer.add_page(page)

    with open(output_pdf, "wb") as f:
        writer.write(f)

    return {
        "removed_text_comments": removed_text,
        "removed_popups": removed_popup,
    }


In [11]:
input_pdf = pick_input_pdf("Select the PDF to remove collapsible comments from")
if input_pdf is None:
    raise RuntimeError("No input PDF selected.")

in_path = Path(input_pdf)
default_out = f"{in_path.stem}_no_comments.pdf"

output_pdf = pick_output_pdf(default_out, "Save the cleaned PDF as")
if output_pdf is None:
    raise RuntimeError("No output path selected.")

stats = remove_collapsible_comments(input_pdf, output_pdf)

msg = (
    f"Done.\n\n"
    f"Input:  {input_pdf}\n"
    f"Output: {output_pdf}\n\n"
    f"Removed Text (sticky note) comments: {stats['removed_text_comments']}\n"
    f"Removed Popup annotations:         {stats['removed_popups']}"
)

print(msg)

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo("PDF annotation removal", msg)
root.destroy()


Done.

Input:  C:/Users/SCHMIDTC18/Downloads/APPT-D-20-00130_reviewer (1).pdf
Output: C:/Users/SCHMIDTC18/Downloads/APPT-D-20-00130_reviewer (1)_no_comments.pdf

Removed Text (sticky note) comments: 54
Removed Popup annotations:         54
